In [ ]:
# %load init_ipynb.py
import os
import sys
import matplotlib.pyplot as pl

sys.path.insert(0, '..')
# assert os.environ["CONDA_DEFAULT_ENV"] == 'wakai'

plotpar = {'axes.labelsize': 16,
           'font.size': 16,
           'legend.fontsize': 16,
           'xtick.labelsize': 16,
           'ytick.labelsize': 16,
           #'text.usetex': False,
           'xtick.direction': 'in',
           'ytick.direction': 'in'
           }
pl.rcParams.update(plotpar)

# known young host stars

In [ ]:
from wakai.planet import get_nexsci_data

df_nexsci = get_nexsci_data(table_name="ps", clobber=False)

In [ ]:
from utils import get_relative_err_index

idx = (df_nexsci.st_age<1) & (df_nexsci.default_flag==1) #& (df_nexsci.disc_facility=='Kepler') 
idx2 = get_relative_err_index(df_nexsci[idx], par='st_age', rel_err=0.2)

df_young = df_nexsci[idx][idx2]
young_hoststar_names = df_young.hostname.unique()
len(young_hoststar_names), young_hoststar_names

# rotation catalogs

In [ ]:
# cd = CatalogDownloader(catalog_name="Rebull2016_Pleiades")
# tabs = cd.get_tables_from_vizier(row_limit=-1)
# df_rebull2016 = tabs[0].to_pandas()
# df_rebull2016 = df_rebull2016.rename({'__V-K_0': 'V-K_0'}, axis=1)
# df_rebull2016.columns

# cd = CatalogDownloader(catalog_name="Douglas2017_Praesepe")
# tabs = cd.get_tables_from_vizier(row_limit=-1)
# df_Douglas2017_Praesepe = tabs[2].to_pandas()
# df_Douglas2017_Praesepe.columns

In [ ]:
from wakai.catalog import CatalogDownloader

cd2 = CatalogDownloader(catalog_name="Lu2024", catalog_type="age")
tabs2 = cd2.get_tables_from_vizier(row_limit=-1)

In [ ]:
df2 = tabs2[0].to_pandas()
df2.head()

## Ruprecht 147 and 5 other well-known clusters

In [ ]:
import numpy as np
from wakai.catalog import CatalogDownloader

cd = CatalogDownloader(catalog_name="Curtis2019_Rup147")
tabs = cd.get_tables_from_vizier(row_limit=-1)
df = tabs[1].to_pandas()
df['logTeff'] = df.Teff.apply(lambda x: np.log10(x))
df['st_rotp'] = df['Prot'].copy()
df = df.rename({"__BP-RP_0": "BP-RP_0"})
df

In [ ]:
df.groupby("Cluster").groups.keys()

## NGC 3532

In [ ]:
cd = CatalogDownloader(catalog_name="Fritzewski2021_NGC3532")
tabs = cd.get_tables_from_vizier(row_limit=-1)
df_ngc3532 = tabs[0].to_pandas()
# df_ngc3532['logTeff'] = df_ngc3532.Teff.apply(lambda x: np.log10(x))
# df_ngc3532['st_rotp'] = df_ngc3532['Prot'].copy()
df_ngc3532.head()

In [ ]:
#query Teff
import warnings
warnings.simplefilter("ignore", category="UnitsWarning")
from wakai.catalog import Target

t = Target(ra_deg=166.95421, dec_deg=-58.65578)
t.query_vizier_param('Teff')#.get('I/345/gaia2')

In [ ]:
try:
    import swifter
except:
    !{sys.executable} -m pip install swifter==1.3.4
    import swifter
assert swifter.__version__ == "1.3.4"

In [ ]:
def get_vizier_param(df_coords, param='Teff', key='I/355/gaiadr3'):
    """try key='I/345/gaia2'
    """
    return df_coords.swifter.apply(lambda coord: Target(*coord, verbose=False)\
            .query_vizier_param(param)\
            .get(key), axis=1)

In [ ]:
import pandas as pd
import warnings
warnings.simplefilter("ignore", category="UnitsWarning")

clobber = False
coords = ['RA_ICRS','DE_ICRS']
catalog_name="Fritzewski2021_NGC3532"
fp = f"../data/{catalog_name}.csv"
if clobber:
    #this cell may take more than 5 minutes for the 1st run
    df_ngc3532['Teff'] = get_vizier_param(df_ngc3532[coords])
    df_ngc3532.to_csv(fp, index=False)
    print("Saved: ", fp)
else:
    df_ngc3532 = pd.read_csv(fp)
    print("Loaded: ", fp)
df_ngc3532.head()

In [ ]:
df_ngc3532['logTeff'] = df_ngc3532.Teff.apply(lambda x: np.log10(x))
df_ngc3532['st_rotp'] = df_ngc3532['Prot'].copy()

## Pisces Eridani

In [ ]:
cd = CatalogDownloader(catalog_name="Curtis2019_PisEri")
tabs = cd.get_tables_from_vizier(row_limit=-1)
df_PisEri = tabs[0].to_pandas()
df_PisEri['logTeff'] = df_PisEri.Teff.apply(lambda x: np.log10(x))
df_PisEri['st_rotp'] = df_PisEri['Prot'].copy()
df_PisEri.head()

In [ ]:
import numpy as np
from matplotlib.ticker import ScalarFormatter
from utils import plot_planets

x, y = 'Teff', 'st_rotp'
df_young['Teff'] = df_young['st_teff'].copy()
df_young['logTeff'] = df_young['st_teff'].apply(lambda x: np.log10(x))

n=1
ax = plot_planets(df_young, x=x, y=y,
                  plot_kwds={'label':'young hosts', 
                            'color': 'k',
                            'zorder': 10, 
                            'marker': '*',
                            'size': 100
                            })
n=2
_ = plot_planets(df_PisEri, x=x, y=y, 
                 ax=ax, 
                 plot_kwds={'marker': '.',
                            'c':'C1', 
                            'label': 'Pis-Eri (0.12 Gyr)'}
                            )
n=3
_ = plot_planets(df_ngc3532, x=x, y=y, 
                 ax=ax, 
                 plot_kwds={'marker': '.',
                            'c':'C2', 
                            'label': 'NGC 3532 (0.30 Gyr)'}
                            )

group_dict = df.groupby('Age', sort=True).groups
for i,age in enumerate(group_dict.keys()):
    idx = group_dict[age]
    d = df.loc[idx]
    name = d.Cluster.iloc[0]
    _ = plot_planets(d, x=x, y=y,
                     ax=ax, 
                     plot_kwds={'marker': '.', 
                                'c':f'C{i+n}', 
                                'label': f'{name} ({age:.2f} Gyr)'}
                                )
ax.legend(fontsize=12, bbox_to_anchor=(1, 0.8))
ax.set_xscale("log")
ax.set_xlabel("$T_{eff}$ (K)")
ax.set_yscale("log")
ax.set_ylabel("$P_{rot}$ (d)")
ax.get_xaxis().set_minor_formatter(ScalarFormatter())
ax.get_yaxis().set_major_formatter(ScalarFormatter())

In [ ]:
fp = "../figures/Prot_Teff.png"
ax.figure.savefig(fp, bbox_inches="tight")